# 3.6 - Sub-clustering the F pathway

Second stage for the retroflection / eastward group. F is ~1% of the archive,
so a single global k-means can only ever give it a handful of clusters - at
K=70 it gets four. Isolating it and re-fitting lets k-means spend all its
clusters inside the pathway instead of on the dense core.

**Selection is by LABEL, not by a geographic box.** Notebook 1.6 gates on the
day-180 endpoint (`lon_180 > -40`), which is a *destination* test: it also
catches C trajectories that loop mid-basin and it drops F members that stop
just short of the line. Now that notebook 03 has labelled every trajectory with
the manual grouping, `cluster_group == F` selects the pathway by its whole
route, which is what the grouping encodes.

1.6 is left untouched; this notebook is the label-based counterpart.

1. Select F from `labeled_trajectories.parquet`.
2. Re-fit k-means inside it, and **save the model immediately**.
3. Grid + overview map to eyeball the sub-clusters.
4. Hand-assign sub-group letters, preview, export.


In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import xarray as xr
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from joblib import Parallel, delayed
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import config as C
import pipeline as P

# --- TUNE ME -----------------------------------------------------------------
GROUP     = "F"      # which merged group to sub-cluster
K_F       = 100      # sub-clusters inside it
COMPLETE_ONLY = True # drop `partial` trajectories (their late days are padded)

# Per-day weights for the SUB-fit. Retroflection lives at day 30-50, so this
# leans early; the parent fit in 1.5 used a different set on purpose.
DAY_WEIGHTS = {30: 2.0, 50: 2.0, 100: 1.0, 150: 1.0, 180: 2.0}

MEMBERS_PER_PANEL = 15
NCOL = 10
# -----------------------------------------------------------------------------

_wd = DAY_WEIGHTS if DAY_WEIGHTS is not None else C.DAY_WEIGHTS
W   = np.array([np.sqrt(_wd[d]) for d in C.DAYS for _ in range(2)])
GROUP_ID = [k for k, v in C.GROUP_NAMES.items() if v == GROUP][0]
print(f"group {GROUP} = id {GROUP_ID} | K_F = {K_F} | weights {_wd}")

group F = id 12 | K_F = 100 | weights {30: 2.0, 50: 2.0, 100: 1.0, 150: 1.0, 180: 2.0}


## Select the group by label
`cluster_group` comes from notebook 03, which applied `config.GROUP_MAP` to the
manual grouping in 1.5. `partial` trajectories are dropped by default: they
reached the gate day but were deleted later, so their last sampled positions
are the *last known fix repeated* and would sit artificially still.

In [2]:
lab = pd.read_parquet(C.LABELED_FILE)
sel = lab.cluster_group == GROUP_ID
if COMPLETE_ONLY:
    sel &= lab.status == "complete"
sub = lab[sel].reset_index(drop=True)

print(f"{GROUP}: {len(sub):,} of {len(lab):,} trajectories "
      f"({100*len(sub)/len(lab):.2f}%)")
print(f"  status  : {sub.status.value_counts().to_dict()}")
print(f"  from parent clusters: {sorted(sub.cluster_label.unique())}")
print(f"  per parent cluster  :")
print(sub.cluster_label.value_counts().sort_index().to_string())
print(f"\n  release years {sub.release_year.min()}-{sub.release_year.max()}, "
      f"{sub.release_year.nunique()} distinct")

F: 179,630 of 15,280,000 trajectories (1.18%)
  status  : {'complete': 179630, 'partial': 0, 'early_loss': 0}
  from parent clusters: [np.int32(32), np.int32(43), np.int32(45), np.int32(58)]
  per parent cluster  :
cluster_label
32    47282
43    52916
45    50594
58    28838

  release years 1993-2013, 21 distinct


## Fit k-means inside the group
Own `StandardScaler` and own weighting - the sub-fit lives in its own space and
its cluster ids have nothing to do with the parent fit's.

In [3]:
X_raw  = P.build_feature_matrix(sub)
scaler = StandardScaler().fit(X_raw)
X      = scaler.transform(X_raw) * W

km = KMeans(n_clusters=K_F, random_state=C.RANDOM_STATE, n_init=10)
labels = km.fit_predict(X)
sizes  = np.bincount(labels, minlength=K_F)
cent_deg = P.centroids_to_degrees(km.cluster_centers_ / W, scaler)
print(f"fit k={K_F} on {len(sub):,} {GROUP} trajectories | "
      f"sizes {sizes.min()}..{sizes.max()}  (median {int(np.median(sizes))})")

fit k=100 on 179,630 F trajectories | sizes 232..3112  (median 1808)


## Save the fit straight away
Notebook 1.5 kept its model only in memory and the grouping had to be
reconstructed later from an archived copy. Not repeating that: the model, its
scaler, its weights and the per-trajectory labels are written now, before any
hand-grouping is anchored to these cluster ids.

In [4]:
MODEL_DIR = C.MODELS_DIR / f"sub_{GROUP}_k{K_F}"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(km, MODEL_DIR / f"kmeans_k{K_F}.pkl")
joblib.dump(scaler, MODEL_DIR / "scaler.pkl")
np.save(MODEL_DIR / "weights_W.npy", W)
np.save(MODEL_DIR / "centroids_deg.npy", cent_deg)
pd.DataFrame({"trajectory_id": sub.trajectory_id.to_numpy(),
              "sub_label": labels.astype(np.int32)}).to_parquet(
    MODEL_DIR / "sub_labels.parquet", index=False)
(MODEL_DIR / "fit_metadata.json").write_text(json.dumps(dict(
    source_notebook="3.6_F_subclustering.ipynb", group=GROUP, group_id=GROUP_ID,
    K=K_F, DAY_WEIGHTS={str(k): v for k, v in _wd.items()}, DAYS=C.DAYS,
    complete_only=COMPLETE_ONLY, RANDOM_STATE=C.RANDOM_STATE,
    n_fitted=int(len(sub)), parent_clusters=sorted(int(x) for x in
                                                   sub.cluster_label.unique()),
    cluster_sizes=sizes.tolist()), indent=2))
print("saved ->", MODEL_DIR)

saved -> /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmean_analysis_30_180_stdZ_w/data/kmeans_models/sub_F_k100


## Cheap coastline
Same trick as the other notebooks: Natural Earth 110 m as plain (lon, lat)
segments drawn on ordinary axes.

In [5]:
import cartopy.feature as cfeature
_coast = []
for geom in cfeature.COASTLINE.with_scale("110m").geometries():
    for line in getattr(geom, "geoms", [geom]):
        _coast.append((np.asarray(line.xy[0]), np.asarray(line.xy[1])))

LON_LIM = (float(np.nanmin(cent_deg[:, 1::2])) - 4,
           float(np.nanmax(cent_deg[:, 1::2])) + 4)
LAT_LIM = (float(np.nanmin(cent_deg[:, 0::2])) - 3,
           float(np.nanmax(cent_deg[:, 0::2])) + 3)

def draw_coast(ax):
    for x, y in _coast:
        ax.plot(x, y, "-", color="0.75", lw=.4, zorder=0)
    ax.set_xlim(*LON_LIM); ax.set_ylim(*LAT_LIM)

def draw_guides(ax):
    for x in (-70, -60, -40):
        ax.axvline(x, linewidth=.5)
    ax.axhline(20, linewidth=.5)
    ax.add_patch(mpatches.Rectangle(xy=(-55, 5), width=10, height=5,
                                    linewidth=1, edgecolor="green",
                                    facecolor="none"))
print("coastline segments:", len(_coast), "| extent", LON_LIM, LAT_LIM)

coastline segments: 134 | extent (-59.30184530958375, -0.9166805744492308) (-3.1690019673892635, 18.605318441610535)


## Stream a few FULL member trajectories per sub-cluster
Look each store up by NAME rather than by `trajectory_id // TRAJ_PER_STORE`:
retiring the duplicated stores renumbered every store index, so the arithmetic
route can point at the wrong file.

In [6]:
N_IO_THREADS = int(os.environ.get("SLURM_CPUS_PER_TASK", 16))
store_paths = {p.stem: p for p in C.list_stores()}

memdf = pd.DataFrame({"trajectory_id": sub.trajectory_id.to_numpy(),
                      "store": sub.store.to_numpy(), "lab": labels})
# explicit loop rather than groupby.apply: the pandas kwargs for keeping the
# group key have changed across versions and silently drop `lab`.
pick = pd.concat(
    [d.sample(min(MEMBERS_PER_PANEL, len(d)), random_state=C.RANDOM_STATE)
     for _, d in memdf.groupby("lab")], ignore_index=True)
pick["local"] = pick.trajectory_id % C.TRAJ_PER_STORE

def _read(name, grp):
    ds  = xr.open_zarr(store_paths[name])
    loc = grp.local.to_numpy()
    return grp.trajectory_id.to_numpy(), ds.lon.values[loc], ds.lat.values[loc]

res = Parallel(n_jobs=N_IO_THREADS, backend="threading")(
    delayed(_read)(name, grp) for name, grp in pick.groupby("store"))
paths = {}
for tids, plon, plat in res:
    for j, tid in enumerate(tids):
        paths[int(tid)] = (plon[j], plat[j])
members = {c: pick.loc[pick.lab == c, "trajectory_id"].to_numpy()
           for c in range(K_F)}


## The grid - one panel per sub-cluster
Blue member paths, orange centroid, the guide lines and the retroflection box.
This is the picture to group from.

In [7]:
nrow = int(np.ceil(K_F / NCOL))
fig, axes = plt.subplots(nrow, NCOL, figsize=(2.9 * NCOL, 2.3 * nrow),
                         squeeze=False, sharex=True, sharey=True)
for c in range(K_F):
    ax = axes[c // NCOL][c % NCOL]
    draw_coast(ax); draw_guides(ax)
    for tid in members[c]:
        lo, la = paths[int(tid)]
        m = ~(np.isnan(lo) | np.isnan(la))
        ax.plot(lo[m], la[m], "-", color="blue", lw=.4, alpha=.3)
    ax.plot(cent_deg[c, 1::2], cent_deg[c, 0::2], ".-", color="darkorange",
            lw=2, zorder=3)
    ax.set_title(f"sub {c}  (n={sizes[c]:,})", fontsize=8)
for j in range(K_F, nrow * NCOL):
    axes[j // NCOL][j % NCOL].axis("off")
fig.suptitle(f"{GROUP} sub-clusters, k={K_F} "
             f"(orange = centroid, blue = members)", y=1.0)
fig.tight_layout()
fig.savefig(f"sub_{GROUP}_K{K_F}_W_{_wd}.pdf", dpi=130, bbox_inches="tight")
plt.show()

## All sub-clusters on one map
Every centroid track together; the number sits at the day-180 endpoint.

In [8]:
fig, ax = plt.subplots(figsize=(11, 6))
draw_coast(ax); draw_guides(ax)
cmap = plt.get_cmap("tab20", K_F)
for c in range(K_F):
    lo, la = cent_deg[c, 1::2], cent_deg[c, 0::2]
    ax.plot(lo, la, "-", color=cmap(c), lw=1.2, alpha=.8, zorder=2)
    ax.text(lo[-1], la[-1], str(c), fontsize=7, ha="center", va="center",
            zorder=4)
ax.set_title(f"All {K_F} {GROUP} sub-cluster centroids "
             f"(day {C.DAYS[0]} -> {C.DAYS[-1]})")
fig.tight_layout(); plt.show()

## Assign each sub-cluster a letter
`SUB_ASSIGN[c] = "letter"`; same letter = merged. Starts all-separate - edit it
after reading the grid, exactly as in notebook 1.5.

In [9]:
import string
def _lbl(c):
    return string.ascii_uppercase[c] if c < 26 else f"g{c}"
SUB_ASSIGN = {c: _lbl(c) for c in range(K_F)}

# --- merge by hand, e.g.: ---
# for c in [0, 3, 7]: SUB_ASSIGN[c] = "F1"

print(f"{len(set(SUB_ASSIGN.values()))} sub-groups (all separate so far)")

100 sub-groups (all separate so far)


## Preview the merged sub-groups
One panel per letter, points coloured by day so the direction of travel is
readable; the label is each sub-cluster's id.

In [10]:
letters = sorted(set(SUB_ASSIGN.values()))
NCOLg = min(5, len(letters))
nrowg = int(np.ceil(len(letters) / NCOLg))
daycmap = plt.get_cmap("viridis", len(C.DAYS))
daycol  = [daycmap(i) for i in range(len(C.DAYS))]

fig, axes = plt.subplots(nrowg, NCOLg, figsize=(3.4 * NCOLg, 2.7 * nrowg),
                         squeeze=False, sharex=True, sharey=True)
for i, L in enumerate(letters):
    ax = axes[i // NCOLg][i % NCOLg]
    draw_coast(ax); draw_guides(ax)
    mem = [c for c in range(K_F) if SUB_ASSIGN[c] == L]
    for c in mem:
        lo, la = cent_deg[c, 1::2], cent_deg[c, 0::2]
        ax.plot(lo, la, "-", color="0.7", lw=1.0, zorder=2)
        ax.scatter(lo, la, c=daycol, s=22, zorder=3, edgecolor="white",
                   linewidths=.4)
        ax.annotate(str(c), (lo[-1], la[-1]), xytext=(4, 4),
                    textcoords="offset points", fontsize=6.5, color="0.1",
                    zorder=5)
    ax.set_title(f"{L}  ({len(mem)} sub-clusters)", fontsize=9)
for j in range(len(letters), nrowg * NCOLg):
    axes[j // NCOLg][j % NCOLg].axis("off")
handles = [plt.Line2D([], [], marker="o", ls="", color=daycol[i], mec="white",
                      ms=7, label=f"day {d}") for i, d in enumerate(C.DAYS)]
fig.legend(handles=handles, loc="lower center", ncol=len(C.DAYS), frameon=False,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"{GROUP} sub-groups (centroids, coloured by day)", y=1.0)
fig.tight_layout(); plt.show()

## Export
Writes the per-trajectory sub-labels next to the model, and prints the maps in
the same form notebook 1.5 exports for `config.py`.

In [11]:
name2id     = {L: i for i, L in enumerate(letters)}
SUB_MAP     = {c: name2id[SUB_ASSIGN[c]] for c in range(K_F)}
SUB_NAMES   = {i: f"{GROUP}-{L}" for L, i in name2id.items()}

out = pd.DataFrame({"trajectory_id": sub.trajectory_id.to_numpy(),
                    "sub_label": labels.astype(np.int32)})
out["sub_group"] = out.sub_label.map(SUB_MAP).astype(np.int32)
OUT = C.DATA_DIR / f"sub_{GROUP}_labels_k{K_F}.parquet"
out.to_parquet(OUT, index=False)
print("saved", out.shape, "->", OUT)

print("\n# weights:", _wd, "| selection: cluster_group ==", GROUP,
      "| complete_only:", COMPLETE_ONLY)
print("SUB_MAP =", SUB_MAP)
print("SUB_NAMES =", SUB_NAMES)
print()
for L in letters:
    mem = [c for c in range(K_F) if SUB_ASSIGN[c] == L]
    print(f"  {L}: {mem}  (n={sum(sizes[c] for c in mem):,})")

saved (179630, 3) -> /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmean_analysis_30_180_stdZ_w/data/sub_F_labels_k100.parquet

# weights: {30: 2.0, 50: 2.0, 100: 1.0, 150: 1.0, 180: 2.0} | selection: cluster_group == F | complete_only: True
SUB_MAP = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18, 19: 19, 20: 20, 21: 21, 22: 22, 23: 23, 24: 24, 25: 25, 26: 26, 27: 27, 28: 28, 29: 29, 30: 30, 31: 31, 32: 32, 33: 33, 34: 34, 35: 35, 36: 36, 37: 37, 38: 38, 39: 39, 40: 40, 41: 41, 42: 42, 43: 43, 44: 44, 45: 45, 46: 46, 47: 47, 48: 48, 49: 49, 50: 50, 51: 51, 52: 52, 53: 53, 54: 54, 55: 55, 56: 56, 57: 57, 58: 58, 59: 59, 60: 60, 61: 61, 62: 62, 63: 63, 64: 64, 65: 65, 66: 66, 67: 67, 68: 68, 69: 69, 70: 70, 71: 71, 72: 72, 73: 73, 74: 74, 75: 75, 76: 76, 77: 77, 78: 78, 79: 79, 80: 80, 81: 81, 82: 82, 83: 83, 84: 84, 85: 85, 86: 86, 87: 87, 88: 88, 89: 89, 90: 90, 91: 91, 92: 9